# Uji Validitas dan Reliabilitas Final Gabungan 29 Butir


## 1. Upload file final

In [1]:
from google.colab import files
up=files.upload()
nama_file=next(iter(up))

Saving Respons Gabungan 3 TI A - Versi Baru - 29 Butir - Jawaban Formulir 1 (6).csv to Respons Gabungan 3 TI A - Versi Baru - 29 Butir - Jawaban Formulir 1 (6).csv


## 2. Persiapan data dan skoring

In [3]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
if "nama_file" not in globals():
    raise ValueError("Jalankan sel upload dan pilih file data terlebih dahulu.")

OUT = Path("hasil_final29")
OUT.mkdir(exist_ok=True)

ekstensi = Path(nama_file).suffix.lower()

if ekstensi == ".csv":
    raw = pd.read_csv(
        nama_file,
        encoding="utf-8-sig",
        sep=None,
        engine="python"
    )

elif ekstensi in {".xlsx", ".xls"}:
    with pd.ExcelFile(nama_file) as book:
        sheets = []

        for sheet in book.sheet_names:
            header = pd.read_excel(book, sheet_name=sheet, nrows=0)
            header.columns = header.columns.astype(str).str.strip()

            if "Kelas" in header.columns:
                sheets.append(sheet)

        if len(sheets) != 1:
            raise ValueError(
                "Periksa file Excel: harus ada satu sheet "
                "respons dengan kolom Kelas."
            )

        raw = pd.read_excel(book, sheet_name=sheets[0])

else:
    raise ValueError("Unggah file .csv, .xlsx, atau .xls.")

raw.columns = raw.columns.astype(str).str.strip()

if "Kelas" not in raw.columns:
    raise ValueError(
        "Kolom Kelas tidak ditemukan. Pastikan file yang diunggah "
        "adalah data respons kuesioner."
    )

kelas = (
    raw["Kelas"]
    .astype(str)
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

data = raw.loc[kelas.eq("3 TI A")].copy()

if data.empty:
    raise ValueError("Tidak ditemukan respons dari kelas 3 TI A.")

cols = [
    c for c in data.columns
    if re.match(r"^.*?\[.+\]\s*$", str(c))
]

assert len(cols) == 29, (
    f"Diharapkan 29 butir gabungan, ditemukan {len(cols)}. "
    "Pastikan menggunakan file respons form gabungan."
)

skala = {
    "Sangat Tidak Setuju (STS)": 1,
    "Tidak Setuju (TS)": 2,
    "Netral (N)": 3,
    "Setuju (S)": 4,
    "Sangat Setuju (SS)": 5
}

num = data[cols].apply(
    lambda s: s.astype(str).str.strip().map(skala)
)

assert num.notna().all().all(), (
    "Ada jawaban kosong/tidak dikenal. "
    "Periksa sumber, jangan mengubah jawaban."
)

assert len(num) > 2, "Jumlah responden harus lebih dari 2."

assert (num.var(ddof=1) > 0).all(), (
    "Ada butir dengan jawaban konstan sehingga korelasi "
    "butir tersebut tidak dapat dihitung."
)

dimensi = {}
kamus = []

for c in cols:
    dim = c.split("[", 1)[0].strip()
    isi = c.split("[", 1)[1].rsplit("]", 1)[0]
    kode, pernyataan = isi.split(".", 1)

    dimensi.setdefault(dim, []).append(c)

    kamus.append({
        "Kode": kode.strip(),
        "Dimensi": dim,
        "Pernyataan": pernyataan.strip(),
        "Kolom": c
    })

K = pd.DataFrame(kamus)
kode_map = dict(zip(K.Kolom, K.Kode))

n = len(num)
t = stats.t.ppf(0.975, n - 2)
r_tabel = t / np.sqrt(n - 2 + t * t)

def alpha(x):
    k = x.shape[1]
    vt = x.sum(axis=1).var(ddof=1)

    if k > 1 and vt > 0:
        return k / (k - 1) * (
            1 - x.var(ddof=1).sum() / vt
        )

    return np.nan


def kategori(a):
    if pd.isna(a):
        return "Tidak dapat dihitung"
    if a >= 0.90:
        return "Sangat baik"
    if a >= 0.80:
        return "Baik"
    if a >= 0.70:
        return "Reliabel"
    return "Tidak reliabel"


print("File terbaca        :", nama_file)
print("Jumlah respons awal :", len(raw))
print("Responden 3 TI A    :", n)
print("Jumlah butir        :", len(cols))
print("Jumlah dimensi      :", len(dimensi))
print("Data kosong         :", int(num.isna().sum().sum()))
print(f"r tabel             : {r_tabel:.6f}")

display(num.head())

File terbaca        : Respons Gabungan 3 TI A - Versi Baru - 29 Butir - Jawaban Formulir 1 (6).csv
Jumlah respons awal : 39
Responden 3 TI A    : 39
Jumlah butir        : 29
Jumlah dimensi      : 5
Data kosong         : 0
r tabel             : 0.316032


,Penggunaan AI pada tugas atau projek kuliah [AI1. Saya menggunakan AI ketika mengerjakan tugas kuliah.],Penggunaan AI pada tugas atau projek kuliah [AI2. Saya menggunakan AI untuk memahami persoalan dalam tugas kuliah.],Penggunaan AI pada tugas atau projek kuliah [AI3. Saya menggunakan AI untuk mencari ide pengerjaan tugas kuliah.],Penggunaan AI pada tugas atau projek kuliah [AI4. Saya menggunakan AI untuk menyusun langkah penyelesaian tugas kuliah.],Penggunaan AI pada tugas atau projek kuliah [AI5. Saya menggunakan AI untuk memperoleh contoh penyelesaian tugas kuliah.],Penggunaan AI pada tugas atau projek kuliah [AI6. Saya menggunakan AI untuk mengembangkan jawaban tugas kuliah.],Penggunaan AI pada tugas atau projek kuliah [AI7. Saya menggunakan AI untuk meninjau jawaban tugas kuliah sebelum dikumpulkan.],Aspek psikologis penggunaan AI pada tugas atau projek kuliah [PS1. Saya merasa membutuhkan bantuan AI untuk mulai mengerjakan tugas kuliah.],Aspek psikologis penggunaan AI pada tugas atau projek kuliah [PS2. Saya merasa perlu memperoleh saran AI sebelum menentukan langkah pengerjaan tugas kuliah.],Aspek psikologis penggunaan AI pada tugas atau projek kuliah [PS3. Saya merasa lebih yakin terhadap jawaban tugas setelah memperoleh konfirmasi dari AI.],...,Kebiasaan penggunaan teknologi digital [TD6. Saya menggunakan jawaban AI dalam tugas kuliah setelah membacanya sekilas.],Prokrastinasi akademik [PA1. Saya menunda memulai tugas kuliah hingga mendekati batas pengumpulan.],Prokrastinasi akademik [PA2. Saya memulai pengerjaan tugas kuliah lebih lambat daripada waktu yang saya rencanakan.],Prokrastinasi akademik [PA3. Saya menunda melanjutkan tugas kuliah yang sudah saya mulai.],Prokrastinasi akademik [PA4. Saya menyelesaikan tugas kuliah pada saat waktu pengumpulan hampir berakhir.],Respons emosional [RE1. Saya merasa tertekan ketika waktu pengerjaan tugas menjadi singkat akibat menunda.],Respons emosional [RE2. Saya merasa cemas ketika tugas yang saya tunda mendekati batas pengumpulan.],Respons emosional [RE3. Saya merasa panik ketika beberapa tugas yang saya tunda harus segera dikumpulkan.],Respons emosional [RE4. Saya merasa gelisah ketika memikirkan tugas yang pengerjaannya saya tunda.],Respons emosional [RE5. Saya merasa kewalahan menghadapi tugas yang menumpuk akibat penundaan.]
0,1,2,4,5,4,2,1,1,2,4,...,2,1,2,4,5,1,2,4,5,4
1,1,2,1,5,5,4,1,1,2,1,...,1,4,4,4,1,5,1,1,5,4
2,1,2,1,2,2,2,1,2,2,1,...,2,1,2,2,1,1,2,2,1,2
3,5,4,2,1,2,4,5,1,2,4,...,5,1,2,5,4,1,2,4,5,4
4,5,4,2,1,2,4,5,5,4,2,...,4,5,4,2,1,5,4,2,1,2


## 3. Pengujian dan grafik

In [4]:
rows=[]; rel=[]; desc=[]; comp=[]
for di,(dim, cs) in enumerate(dimensi.items(),1):
    x = num[cs]; total=x.sum(axis=1); a=alpha(x)
    rel.append({'Dimensi':dim, 'Butir':len(cs), 'Alpha':a, 'Kategori':kategori(a)})
    skor=x.mean(axis=1)
    desc.append({'Dimensi':dim, 'Mean':skor.mean(), 'SD':skor.std(ddof=1),
                 'Skewness':stats.skew(skor,bias=False),
                 'Kurtosis_excess':stats.kurtosis(skor,fisher=True,bias=False)})
    for c in cs:
        r,p=stats.pearsonr(x[c],total)
        rc,_=stats.pearsonr(x[c],total-x[c])
        rows.append({'Dimensi':dim,'Kode':kode_map[c],'r_hitung':r,'r_tabel':r_tabel,
                     'Corrected_r':rc,'p_value':p,'Alpha_dihapus':alpha(x.drop(columns=c)),
                     'Keputusan':'Valid' if r>r_tabel else 'Tidak valid'})
    counts=x.stack().value_counts().reindex(range(1,6),fill_value=0)
    for nilai,jumlah in counts.items():
        comp.append({'Dimensi':dim,'Skor':nilai,'Jumlah':int(jumlah),'Persen':100*jumlah/x.size})
    fig,ax=plt.subplots(figsize=(7,2.7))
    bars=ax.bar(['STS','TS','N','S','SS'],counts.values,color='#737373')
    for bar,v in zip(bars,counts.values):
        ax.text(bar.get_x()+bar.get_width()/2,v+.5,f'{v} ({100*v/x.size:.1f}%)',ha='center',fontsize=9)
    ax.set_ylim(0,counts.max()*1.25);ax.set_ylabel('Jumlah jawaban')
    ax.set_title(dim);fig.tight_layout();fig.savefig(OUT/f'dimensi_{di}.png',dpi=160);plt.close(fig)
V=pd.DataFrame(rows);R=pd.DataFrame(rel);S=pd.DataFrame(desc);C=pd.DataFrame(comp)



## 4. Ekspor dan hasil lengkap

In [9]:
daftar_tabel = [
    ("validitas", V),
    ("reliabilitas", R),
    ("deskriptif", S),
    ("komposisi", C),
    ("kamus_butir", K),
    ("skor_likert", num.rename(columns=kode_map))
]

for name, table in daftar_tabel:
    table.to_csv(
        OUT / f"{name}.csv",
        index=False,
        encoding="utf-8-sig"
    )

print(f"File: {nama_file}")
print(
    f"n={n} | butir={len(cols)} | "
    f"dimensi={len(dimensi)}"
)
print(
    f"Data kosong/gagal konversi: {int(num.isna().sum().sum())} | "
    f"df={n - 2} | r tabel={r_tabel:.6f}"
)

print("\nHASIL VALIDITAS")
print(V.to_string(index=False))

print("\nHASIL RELIABILITAS")
print(R.to_string(index=False))

print("\nSTATISTIK DESKRIPTIF")
print(S.to_string(index=False))

print(
    "\nValid:",
    int(V["Keputusan"].eq("Valid").sum()),
    "dari",
    len(V)
)

print(
    "Corrected r >= 0.30:",
    int((V["Corrected_r"] >= 0.30).sum()),
    "dari",
    len(V)
)

print("\nTabel berhasil disimpan di:", OUT.resolve())



File: Respons Gabungan 3 TI A - Versi Baru - 29 Butir - Jawaban Formulir 1 (6).csv
n=39 | butir=29 | dimensi=5
Data kosong/gagal konversi: 0 | df=37 | r tabel=0.316032

HASIL VALIDITAS
                                                     Dimensi Kode  r_hitung  r_tabel  Corrected_r      p_value  Alpha_dihapus   Keputusan
                 Penggunaan AI pada tugas atau projek kuliah  AI1  0.661743 0.316032     0.423657 4.491424e-06       0.563319       Valid
                 Penggunaan AI pada tugas atau projek kuliah  AI2  0.514315 0.316032     0.346613 8.096739e-04       0.595235       Valid
                 Penggunaan AI pada tugas atau projek kuliah  AI3  0.678301 0.316032     0.517242 2.085200e-06       0.540046       Valid
                 Penggunaan AI pada tugas atau projek kuliah  AI4  0.237517 0.316032    -0.029939 1.454024e-01       0.705528 Tidak valid
                 Penggunaan AI pada tugas atau projek kuliah  AI5  0.538916 0.316032     0.310607 4.006602e-04       0.602963

## 5. Simulasi penghapusan dan sampel

In [10]:
c = K.loc[K.Kode.eq('AI4'),'Kolom'].iloc[0]
dim=K.loc[K.Kode.eq('AI4'),'Dimensi'].iloc[0]
x=num[dimensi[dim]]; setelah=x.drop(columns=c)
print('Simulasi AI4: alpha awal=',alpha(x),'alpha sesudah=',alpha(setelah))
for c2 in setelah:
    rr,pp=stats.pearsonr(setelah[c2],setelah.sum(axis=1))
    print(kode_map[c2],rr,pp,'Valid' if rr>r_tabel else 'Tidak valid')
sub=num.sample(n=min(20,n),random_state=1);ns=len(sub)
t2=stats.t.ppf(.975,ns-2);rt2=t2/np.sqrt(ns-2+t2*t2)
vsub=sum(stats.pearsonr(sub[c],sub[cs].sum(axis=1))[0]>rt2 for cs in dimensi.values() for c in cs)
print(f'Subset acak n={ns}, random_state=1: r tabel={rt2:.6f}; valid={vsub}/{len(cols)}')


Simulasi AI4: alpha awal= 0.6297145769622836 alpha sesudah= 0.7055283525509399
AI1 0.817859485336596 2.0713684557466207e-10 Valid
AI2 0.48267235115297513 0.0018579550649112505 Valid
AI3 0.6656116048241213 3.7703455646319224e-06 Valid
AI5 0.4635753450483548 0.0029561701370949974 Valid
AI6 0.4118241224389239 0.009190240331523353 Valid
AI7 0.8971778705568021 1.0665155811815775e-14 Valid
Subset acak n=20, random_state=1: r tabel=0.443763; valid=21/29


## 6. Unduh hasil

In [8]:
import shutil
shutil.make_archive("Hasil_Final29", "zip", OUT)
files.download("Hasil_Final29.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Kesimpulan
Sebanyak 26 butir valid dan 3 tidak valid (AI4, PS4, TD6); RE4 valid. Kelima dimensi awal belum reliabel. Simulasi menghapus AI4 menghasilkan alpha penggunaan AI 0,705528 dan seluruh enam butir tersisa valid pada sampel yang sama. Hasil utama tetap melaporkan 29 butir.